In [1]:
#!/usr/bin/env python
# coding: utf-8

import json
import argparse
import os
import tqdm
import pandas as pd
import time
from openai import OpenAI

from dotenv import load_dotenv
load_dotenv()

import base64

def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")

In [2]:
def load_template(template_dir, template_name):
    template_path = os.path.join(template_dir, f'prompt-{template_name}.txt')
    if not os.path.exists(template_path):
        raise FileNotFoundError(f"Template not found: {template_path}")
    with open(template_path, 'r', encoding='utf-8') as f:
        template_text = f.read()
    return template_text

In [3]:
def get_instruction_text(template_dir, base_template_name, story_images, character_images=None, character_names=None):
    if character_images and len(character_images) > 0:
        template_name = f"{base_template_name}-w-names"
    else:
        template_name = f"{base_template_name}-wo-names"

    template = load_template(template_dir, template_name)

    fill_values = {
        'story_images': story_images,
        'num_story_images': len(story_images)
    }

    if character_images:
        fill_values['character_images'] = character_images
        fill_values['num_character_images'] = len(character_images)
        fill_values['character_images_text'] = (
            '1 character image' if len(character_images) == 1 else f"{len(character_images)} character images"
        )
    else:
        fill_values['character_images'] = []
        fill_values['num_character_images'] = 0
        fill_values['character_images_text'] = 'no character images'

    if character_names:
        fill_values['character_names'] = character_names
        fill_values['character_names_text'] = ', '.join(character_names)
    else:
        fill_values['character_names'] = []
        fill_values['character_names_text'] = 'no character names'

    filled_instruction = template.format(**fill_values)

    print('FILLED INSTRUCTION', filled_instruction)
    
    return filled_instruction

In [4]:
import numpy as np
def split_dataframe(df, random_seed=42):
    np.random.seed(random_seed)
    
    grouped = df.groupby('story_id')
    
    df1_rows = []
    df2_rows = []
    df3_rows = []

    print(grouped)
    
    for story_id, group in grouped:
        assert len(group) == 3, f"Story ID {story_id} does not have exactly 3 rows."
        
        group = group.sample(frac=1, random_state=random_seed)
        
        df1_rows.append(group.iloc[0])
        df2_rows.append(group.iloc[1])
        df3_rows.append(group.iloc[2])
    
    return pd.DataFrame(df1_rows).reset_index(drop=True), pd.DataFrame(df2_rows).reset_index(drop=True), pd.DataFrame(df3_rows).reset_index(drop=True)

In [ ]:
def run_new(df,
        output_dir,
        prompt_dir,
        prompt_name,
        model,
        client,
        seed):
    """
    Updated run function that constructs image paths from story_id.
    Images are named as {story_id}_img{i}.jpg and {story_id}_char{i}.jpg
    """

    if prompt_name == "large":
        output_dir = os.path.join(output_dir, f"seed-{seed}")
    os.makedirs(output_dir, exist_ok=True)
    
    if prompt_name == "large":
        system_prompt = (
            "You are a helpful assistant and an experienced expert crowdworker. "
            "You are qualified to perform the following task. The title of the task you are working on is: "
            "\"Help us bridge the gap between AI and humans in telling stories about movies!\" "
            "The task description is as follows: we are a group of researchers working with large language models, "
            "and we ask for your help in collecting stories based on the images provided. The data you submit will "
            "be used to build and improve AI models that understand how to generate stories about movies just as you do! "
            "We're very excited to have you join our experiment! Please carefully read the instructions. "
            "You must follow all instructions in order to be eligible for payment."
        )
    else:
        system_prompt = (
            "You are a helpful assistant and an experienced expert crowdworker. "
            "You are qualified to perform the following task."
        )

    for idx, row in tqdm.tqdm(df.iterrows(), total=len(df)):
        
        story_id = row['story_id']

        # If the output file already exists, skip this story
        output_file = os.path.join(output_dir, f"{story_id}.parquet")
        if os.path.exists(output_file):
            tqdm.tqdm.write(f"Skipping story_id {story_id}: output already exists at {output_file}")
            continue

        basepath_images = '../../data/sampled_60/images/'
        basepath_character_images = '../../data/sampled_60/characters/'

        # Construct story image paths based on story_id
        story_images = []
        for i in range(10):  # Check for img0 through img9
            img_path = f'{basepath_images}{story_id}_img{i}.jpg'
            if os.path.exists(img_path):
                story_images.append(img_path)
        
        # Construct character image paths and extract character names
        character_images = []
        character_names = []
        for i in range(5):  # Check for char0 through char4
            char_col = f'char{i}'
            char_path = f'{basepath_character_images}{story_id}_char{i}.jpg'
            
            # Check if character name exists and is not empty
            if char_col in row and pd.notna(row[char_col]) and row[char_col] not in [None, '', '{}']:
                character_names.append(row[char_col])
                
                # Check if corresponding character image file exists
                if os.path.exists(char_path):
                    character_images.append(char_path)
        
        # Convert to None if empty
        character_images = character_images if len(character_images) > 0 else None
        character_names = character_names if len(character_names) > 0 else None

        if character_images is not None:
            all_images = story_images + character_images
        else:
            all_images = story_images

        print('all images', all_images)

        instruction_text = get_instruction_text(
            prompt_dir,
            base_template_name=prompt_name,
            story_images=story_images,
            character_images=character_images,
            character_names=character_names
        )
    
        messages = [
            {
                "role": "system",
                "content": [
                    {
                        "type": "text",
                        "text": system_prompt,
                    }
                ],
            },
            {
                "role": "user",
                "content": [{"type": "text", "text": instruction_text}],
            },
        ]

    
        for image_path in all_images:
                
            base64_image = encode_image(image_path)
            
            if os.path.exists(image_path):
                messages.append(
                    {
                        "role": "user",
                        "content": [
                            {
                                "type": "image_url",
                                "image_url": {"url": f"data:image/jpeg;base64,{base64_image}"},
                            }
                        ],
                    }
                )

        try:

            if prompt_name == "large" and seed == 42:

                # SPEAKER 1
                response = client.chat.completions.create(
                    model = model,

                    temperature = 0.6,
                    top_p = 0.95,

                    seed = seed,
                    messages = messages,
                    max_completion_tokens = 4096
                )
            

            if prompt_name == "large" and seed == 43:

                # SPEAKER 2
                response = client.chat.completions.create(
                    model = model,

                    temperature = 0.6,
                    top_p = 0.95,

                    seed = seed,
                    messages = messages,
                    max_completion_tokens = 4096
                )

            if prompt_name == "large" and seed == 44:

                # SPEAKER 3
                response = client.chat.completions.create(
                    model = model,

                    temperature = 0.6,
                    top_p = 0.95,

                    seed = seed,
                    messages = messages,
                    max_completion_tokens = 4096
                )

            else:

                # medium or original prompts, no "multiple speakers"
                response = client.chat.completions.create(
                    model = model,

                    temperature = 0.6,
                    top_p = 0.95,

                    seed = 42,
                    messages = messages,
                    max_completion_tokens = 4096
                )
                    
            model_out = response.choices[0].message.content
            model = response.model
            system_fingerprint = response.system_fingerprint

        except Exception as e:
            tqdm.tqdm.write(f'Failed on story_id {story_id}: {e}')
            continue

        output_file = os.path.join(output_dir, f"{story_id}.parquet")

        result_df = pd.DataFrame([{ 
            "story_id": story_id,
            "instruction_text": instruction_text,
            "model_output": model_out,
            "model": model,
            "system_fingerprint": system_fingerprint,
            "seed": seed
        }])

        result_df.to_parquet(output_file, index=False)
        
        tqdm.tqdm.write(f"Saved output for story_id {story_id} -> {output_file}")

In [ ]:
input_file = '../../data/sampled_60/sampled_60_stories.json'
output_dir = './out-gpt4o-60stories/'
template_dir = '../../data/prompts/'
model_path = 'gpt-4o-2024-08-06'

template_name = 'large'

In [7]:
import numpy as np
def split_dataframe_fixed(df, random_seed=42, expected_rows_per_story=None):
    """
    Split dataframe based on story_id groups.
    
    Args:
        df: Input dataframe
        random_seed: Random seed for sampling
        expected_rows_per_story: Expected number of rows per story. If None, auto-detect.
    """
    np.random.seed(random_seed)
    
    grouped = df.groupby('story_id')
    
    # Auto-detect expected rows per story if not provided
    if expected_rows_per_story is None:
        story_counts = df['story_id'].value_counts()
        expected_rows_per_story = story_counts.iloc[0]  # Use the count from first story
        print(f"Auto-detected {expected_rows_per_story} rows per story")
    
    df1_rows = []
    df2_rows = []
    df3_rows = []


    print(f"Processing {len(grouped)} story groups...")
    
    for story_id, group in grouped:
        actual_rows = len(group)
        
        if actual_rows != expected_rows_per_story:
            raise ValueError(f"Story ID {story_id} has {actual_rows} rows, expected {expected_rows_per_story}")
        
        group = group.sample(frac=1, random_state=random_seed)
        
        if expected_rows_per_story == 1:
            # For original/medium prompts: all stories go to df1, df2 and df3 are empty
            df1_rows.append(group.iloc[0])
        elif expected_rows_per_story == 3:
            # For large prompt: distribute across df1, df2, df3
            df1_rows.append(group.iloc[0])
            df2_rows.append(group.iloc[1])
            df3_rows.append(group.iloc[2])
        else:
            raise ValueError(f"Unsupported number of rows per story: {expected_rows_per_story}")
    
    return (pd.DataFrame(df1_rows).reset_index(drop=True), 
            pd.DataFrame(df2_rows).reset_index(drop=True), 
            pd.DataFrame(df3_rows).reset_index(drop=True))

In [8]:
import numpy as np

df = pd.read_json(input_file, lines=True)

# Use the fixed function that handles both original/medium (1 row per story) and large (3 rows per story)
df1, df2, df3 = split_dataframe_fixed(df, random_seed=42)

seeds = [44] if template_name in ['original'] else [42, 43, 44]

Auto-detected 1 rows per story
Processing 60 story groups...


In [9]:
seeds = [42, 43, 44]
print(seeds)

[42, 43, 44]


In [ ]:
final_output_dir = os.path.join(output_dir, f'prompt-{template_name}-outputs')
os.makedirs(final_output_dir, exist_ok=True)

# Load API key from config file
with open('config.json', 'r') as f:
    config = json.load(f)

client = OpenAI(api_key=config['openai_api_key'])

if template_name in ['original']:
    print(f"Running with template: {template_name}, only df1, seed {seeds[0]}")
    print(df1.shape)

    run_new(df1,
        final_output_dir,
        template_dir,
        template_name,
        model_path,
        client,
        seed=seeds[0]
       )


elif template_name == 'large':
    print(f"Running with template: {template_name}, df1, df2, df3 separately")

    for num, single_df in enumerate([df1, df2, df3]):
        print(single_df.shape)
        current_seed = seeds[num]

        run_new(single_df,
            final_output_dir,
            template_dir,
            template_name,
            model_path,
            client,
            seed=current_seed
            )

Running with template: large, df1, df2, df3 separately
(60, 41)


100%|██████████| 60/60 [00:00<00:00, 788.75it/s]


Skipping story_id 345: output already exists at ./out-gpt4o-60stories/prompt-large-outputs/seed-42/345.parquet
Skipping story_id 515: output already exists at ./out-gpt4o-60stories/prompt-large-outputs/seed-42/515.parquet
Skipping story_id 721: output already exists at ./out-gpt4o-60stories/prompt-large-outputs/seed-42/721.parquet
Skipping story_id 723: output already exists at ./out-gpt4o-60stories/prompt-large-outputs/seed-42/723.parquet
Skipping story_id 733: output already exists at ./out-gpt4o-60stories/prompt-large-outputs/seed-42/733.parquet
Skipping story_id 948: output already exists at ./out-gpt4o-60stories/prompt-large-outputs/seed-42/948.parquet
Skipping story_id 1111: output already exists at ./out-gpt4o-60stories/prompt-large-outputs/seed-42/1111.parquet
Skipping story_id 1201: output already exists at ./out-gpt4o-60stories/prompt-large-outputs/seed-42/1201.parquet
Skipping story_id 1214: output already exists at ./out-gpt4o-60stories/prompt-large-outputs/seed-42/1214.par

100%|██████████| 60/60 [00:00<00:00, 821.96it/s]


Skipping story_id 345: output already exists at ./out-gpt4o-60stories/prompt-large-outputs/seed-43/345.parquet
Skipping story_id 515: output already exists at ./out-gpt4o-60stories/prompt-large-outputs/seed-43/515.parquet
Skipping story_id 721: output already exists at ./out-gpt4o-60stories/prompt-large-outputs/seed-43/721.parquet
Skipping story_id 723: output already exists at ./out-gpt4o-60stories/prompt-large-outputs/seed-43/723.parquet
Skipping story_id 733: output already exists at ./out-gpt4o-60stories/prompt-large-outputs/seed-43/733.parquet
Skipping story_id 948: output already exists at ./out-gpt4o-60stories/prompt-large-outputs/seed-43/948.parquet
Skipping story_id 1111: output already exists at ./out-gpt4o-60stories/prompt-large-outputs/seed-43/1111.parquet
Skipping story_id 1201: output already exists at ./out-gpt4o-60stories/prompt-large-outputs/seed-43/1201.parquet
Skipping story_id 1214: output already exists at ./out-gpt4o-60stories/prompt-large-outputs/seed-43/1214.par

...

Saved output for story_id 12423 -> ./out-gpt4o-60stories/prompt-large-outputs/seed-44/12423.parquet
all images ['../../data/60_images/images/13596_img0.jpg', '../../data/60_images/images/13596_img1.jpg', '../../data/60_images/images/13596_img2.jpg', '../../data/60_images/images/13596_img3.jpg', '../../data/60_images/images/13596_img4.jpg', '../../data/60_images/characters/13596_char0.jpg']
FILLED INSTRUCTION Describe a sequence of 5 story images as one continuous story. Each time you begin describing a new image, insert [SEP] before the description. Story is a set of descriptions about multiple images. The descriptions must depict the content of the images and together form a coherent whole. They must be accurate and descriptive of the visual story, but not unnecessarily long. A sequence of images is a number of images extracted from movies and curated in an order that is coherent. Images include characters that appear in the story. Characters are the list of images of movie characters

 98%|█████████▊| 59/60 [04:30<00:10, 10.80s/it]

Saved output for story_id 13596 -> ./out-gpt4o-60stories/prompt-large-outputs/seed-44/13596.parquet
all images ['../../data/60_images/images/13683_img0.jpg', '../../data/60_images/images/13683_img1.jpg', '../../data/60_images/images/13683_img2.jpg', '../../data/60_images/images/13683_img3.jpg', '../../data/60_images/images/13683_img4.jpg', '../../data/60_images/images/13683_img5.jpg', '../../data/60_images/characters/13683_char0.jpg', '../../data/60_images/characters/13683_char1.jpg', '../../data/60_images/characters/13683_char2.jpg', '../../data/60_images/characters/13683_char3.jpg']
FILLED INSTRUCTION Describe a sequence of 6 story images as one continuous story. Each time you begin describing a new image, insert [SEP] before the description. Story is a set of descriptions about multiple images. The descriptions must depict the content of the images and together form a coherent whole. They must be accurate and descriptive of the visual story, but not unnecessarily long. A sequence of

100%|██████████| 60/60 [04:36<00:00,  4.61s/it]

Saved output for story_id 13683 -> ./out-gpt4o-60stories/prompt-large-outputs/seed-44/13683.parquet
